In [ ]:
from pathlib import Path
import os
import torch

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    try:
        drive.mount("/content/drive")
    except Exception as e:                      # Colab credential-propagation bug, Nov 2026
        print("Drive mount failed, using local disk:", e)
    REPO, BRANCH = Path("/content/motionnet_cnn"), "main"
    if REPO.exists():
        !cd {REPO} && git fetch -q origin && git checkout -q {BRANCH} && git pull -q origin {BRANCH}
    else:
        !git clone -b {BRANCH} https://github.com/guardomayas/motionnet.git {REPO}
    !pip install -q -e {REPO} pyyaml
    import site, importlib
    site.main(); importlib.invalidate_caches()

from motionnet.paths import setup, compose, Paths
from motionnet.dataset import get_data, batches
import seaborn as sns
sns.set_context("talk")

if IN_COLAB and not os.path.ismount("/content/drive"):
    base = Path("/content/NUIN")                 # local, NOT persistent: download results at the end
    P = Paths(corpus=base / "van_hateren", coeffs=base / "motionnet/cache",
              movies=Path("/content/movie_cache"), ckpt=base / "motionnet/ckpt")
    for d in (P.coeffs, P.movies, P.ckpt):
        d.mkdir(parents=True, exist_ok=True)
    assert P.corpus.exists(), f"put the van Hateren .iml files in {P.corpus} (gdown --folder ...)"
else:
    P = setup(IN_COLAB)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| corpus:", P.corpus)

In [ ]:
EXP = "s04_backend_noise"
POINT = {"loss.gamma_var": 0.5, "model.layer2_noise_std": 0.1, "optim.seed": 0, "model.rf_size_1": 5, "loss.lambda_r2": 0.1, "model.rf_size_2": 7}

# compose() checks every key, so a stale name fails here instead of training silently
cfg, grid = compose(exp_name=EXP, corpus=P.corpus, **POINT)
TAG = "_".join(f"{k.split('.')[-1]}-{v}" for k, v in POINT.items())   # same as sweep.tag_for
OUT = Path(P.ckpt) /"adhoc"

data = get_data(cfg["stimulus"], P.movies, device)
info = data.info
print(TAG)
print("train / val movies:", info.n_train, info.n_val, "| fps", info.fps, "| sigma_px", round(info.sigma_px, 2))
print("model:", cfg["model"])
print("loss: ", cfg["loss"])

## Sanity check: overfit one batch

If R² doesn't climb well above 0 here, something is wrong with data or model, and the full run will be wasted.

In [ ]:
from motionnet.models import MotionCNN
from motionnet.train.loop import compute_v_std

probe = MotionCNN(fps=info.fps, **cfg["model"]).to(device)
opt = torch.optim.AdamW(probe.parameters(), lr=1e-3)
valid = slice(info.gray_frames + probe.max_delay - 1, None)
v_std = compute_v_std(data.train, valid, device)

batch = next(iter(batches(data.train, 32, shuffle=True, drop_last=True)))
movie, target = batch["movie"].to(device), batch["vel_deg_s"].to(device) / v_std
t = target[:, valid]
for step in range(301):
    p = probe(movie)[:, valid]
    loss = (p - t).pow(2).sum(-1).mean()
    opt.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(probe.parameters(), 1.0); opt.step()
    if step % 100 == 0:
        r2 = 1 - (p - t).detach().pow(2).mean((0, 1)) / t.var((0, 1))
        print(f"{step:4d}  loss {loss.item():.3f}  R2 x {r2[0]:.3f}  y {r2[1]:.3f}")
del probe, opt
if device.type == "cuda":
    torch.cuda.empty_cache()

## Module based training loop

In [ ]:
from motionnet.train import fit

if (OUT / f"{TAG}.pt").exists():
    print("already trained, loading")
    from motionnet.train.loop import load_checkpoint
    model, ck = load_checkpoint(OUT / f"{TAG}.pt", device, info)
    history = ck["history"]
else:
    model, history, ckpt = fit(cfg, data, device, OUT, tag=TAG, log_every=10)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

tr, va = pd.DataFrame(history["train"]), pd.DataFrame(history["val"])
ep = range(1, len(tr) + 1)
fig, ax = plt.subplots(1, 4, figsize=(18, 4))

ax[0].plot(ep, tr["nmse"], label="train"); ax[0].plot(ep, va["nmse"], label="val")
ax[0].axhline(1, ls="--", c="k", lw=1, label="predict zero")
ax[0].set(xlabel="epoch", ylabel="nmse")
ax[1].plot(ep, va["r2_x"], label="$v_x$"); ax[1].plot(ep, va["r2_y"], label="$v_y$")
ax[1].set(xlabel="epoch", ylabel="val R²", ylim=(0, 1))
ax[2].plot(ep, va["svar"], label="$V_{sp}$"); ax[2].plot(ep, va["mse"], label="pooled MSE")
ax[2].set(xlabel="epoch", ylabel="loss terms")
ax[3].plot(ep, va["rate1"], label="layer 1"); ax[3].plot(ep, va["rate2"], label="layer 2")
ax[3].set(xlabel="epoch", ylabel="rate (sum over channels)")
for a in ax:
    a.legend(frameon=False)
L = cfg["loss"]; M = cfg["model"]
fig.suptitle(f"γ_var={L['gamma_var']:g}  λ_r1={L['lambda_r1']:g}  λ_r2={L['lambda_r2']:g}  "
             f"σ_in={M['input_noise_std']:g}  σ₁={M['layer_noise_std']:g}  σ₂={M['layer2_noise_std']:g}")
fig.tight_layout()

# Diagnostics

In [ ]:
model.eval()
with torch.no_grad():
    vb = next(iter(batches(data.val, 32)))
    model(vb["movie"].to(device))

g, th = model.gain.detach().cpu(), model.threshold.detach().cpu()
k2 = model.layer2.weight.detach().cpu()
k2 = k2 / k2.flatten(1).norm(dim=1)[:, None, None, None]
mass = k2.pow(2).sum(dim=(0, 2, 3))                      # layer-2 weight onto each layer-1 channel
print("layer 1:  ch   gain   thresh  active  L2-mass")
for c in range(len(g)):
    print(f"          {c+1:2d}  {g[c]:6.3f}  {th[c]:6.3f}  {model._active1[c]:6.3f}  {mass[c]:6.3f}")

h, th2 = model.gain2.detach().cpu(), model.threshold2.detach().cpu()
R = model.readout.detach().cpu(); R = R / R.norm(dim=1, keepdim=True)
w = R.pow(2).sum(0)                                      # readout weight per layer-2 unit
print("\nlayer 2:  m   gain   thresh  active  readout w")
for m in range(len(h)):
    print(f"          {m+1:2d}  {h[m]:6.3f}  {th2[m]:6.3f}  {model._active2[m]:6.3f}  {w[m]:6.3f}")

In [ ]:
import numpy as np
from scipy.ndimage import gaussian_filter
import torch.nn as nn
def temporal_kernels(cnn):
    """(N1, D) kernels in plot orientation: lag 0 LAST, matching t_ms."""
    phi = cnn.temporal_w @ cnn.tbasis.T                 # (N1, D), lag 0 first
    phi = phi / (phi.norm(dim=1, keepdim=True) + 1e-8)  # same as forward()
    return phi.flip(-1).detach().cpu().numpy()          # lag 0 last


def effective_rf(cnn, info, up=None):
    """W1 as point masses on the source-px grid, blurred by the acceptance Gaussian."""
    up = up or int(round(info.spacing_px))                # px per tap (13)
    sw = cnn.spatial_kernel[:, 0].detach().cpu().numpy()  # (N1, k, k)
    n, k, _ = sw.shape
    pad = int(np.ceil(3 * info.sigma_px))
    size = k * up + 2 * pad
    comb = np.zeros((n, size, size))
    c = pad + up // 2 + np.arange(k) * up                 # tap centres in px
    comb[:, c[:, None], c[None, :]] = sw
    eff = np.stack([gaussian_filter(m, info.sigma_px, mode="constant") for m in comb])
    return eff, info.deg_per_tap / info.spacing_px        # map, deg per px


def plot_cnn_subunits_1(cnn, info, effective=True):
    n_su, max_delay = cnn.num_subunits_1, cnn.max_delay
    tw = temporal_kernels(cnn)
    w1 = cnn.spatial_kernel[:, 0].detach().cpu().numpy().copy()

    if effective:
        sw, deg_per_px = effective_rf(cnn, info)
        half = sw.shape[-1] * deg_per_px / 2
        label = f"W₁ ⊛ acceptance  (σ = {info.sigma_px * deg_per_px:.2f}°)"
    else:
        sw = w1.copy()
        half = sw.shape[-1] * info.deg_per_tap / 2
        label = "W₁ (weights only)"

    for n in range(n_su):                   # sign from W1, applied to both factors
        if abs(w1[n].min()) > w1[n].max():
            sw[n], tw[n] = -sw[n], -tw[n]
        s = np.linalg.norm(w1[n])           # ≈1 (unit-norm), kept for safety
        sw[n] /= s
        tw[n] *= s

    extent = [-half, half, -half, half]
    t_ms = np.arange(-max_delay + 1, 1) * (1000.0 / info.fps)
    vlim, ylim = np.abs(sw).max(), np.abs(tw).max()

    fig, axs = plt.subplots(n_su, 3, figsize=(8, 3.2 * n_su), squeeze=False,
                            gridspec_kw=dict(width_ratios=[1, 1.9, 0.1]))
    for n in range(n_su):
        axs[n, 0].plot(t_ms, tw[n])
        axs[n, 0].axhline(0, ls=":", c="k", lw=0.8)
        axs[n, 0].set_ylim(-ylim, ylim)
        axs[n, 0].tick_params(labelbottom=(n == n_su - 1))
        im = axs[n, 1].imshow(sw[n], cmap="RdBu_r", vmin=-vlim, vmax=vlim,
                              extent=extent, interpolation="nearest")
        axs[n, 1].set_title(f"subunit 1,{n+1}")
        plt.colorbar(im, cax=axs[n, 2])
    axs[-1, 0].set_xlabel(r"$\Delta t$ [ms]")
    axs[-1, 1].set_xlabel("azimuth [deg]")
    fig.suptitle(label)
    fig.tight_layout()
    return fig, axs


def plot_cnn_subunits_2(cnn: nn.Module):
    """
    Plot the second layer of subunits of a CNN.
    """
    cnn_filters_2 = cnn.layer2.weight.to("cpu").detach()

    fig, axs = plt.subplots(cnn.num_subunits_2,
                            cnn.num_subunits_1,
                            figsize=(4 * cnn.num_subunits_2,
                                    4 * cnn.num_subunits_1),
                            sharex=True, sharey=True)
    # vlim = abs(cnn_filters_2).max()
    percentile = 99
    vlim = torch.quantile(cnn_filters_2.abs().flatten(), percentile / 100).item()
    for i in range(cnn.num_subunits_2):
        for j in range(cnn.num_subunits_1):
            axs[i, j].imshow(cnn_filters_2[i, j],
                            vmin=-vlim, vmax=vlim, cmap="RdBu_r")

            axs[i, j].set_title('subunit 1,{} $\\to$ 2,{}'.format(j+1,i+1))


In [ ]:
plot_cnn_subunits_1(model, info, effective=True)

In [ ]:
plot_cnn_subunits_2(model)

In [ ]:
from motionnet.models.cnn import unit
import numpy as np, pandas as pd

R = unit(model.readout, (1,)).detach().cpu()               # rows (v_x, v_y), unit norm as in forward()
df = pd.DataFrame(R.T.numpy(), columns=["R_x", "R_y"],
                  index=[f"2,{m+1}" for m in range(R.shape[1])])
df["w"] = df.R_x**2 + df.R_y**2                           # weight in the readout (matches the diagnostics)
df["angle_deg"] = np.degrees(np.arctan2(df.R_y, df.R_x)).round(0)
print(df.round(3))

In [ ]:
# """"
# $$RF_m(x,y,t) = \sum_c g_c\,(K_{mc} * S_c)(x,y)\,\varphi_c(t)$$\
# """"   
from scipy.signal import convolve2d
import matplotlib.pyplot as plt

def layer2_st_rf(model):
    """Linearised space-time RF of each layer-2 unit in receptor coordinates.
    Exact only where layer 1 is linear (ReLU inactive); ignores thresholds."""
    S   = unit(model.spatial_kernel, (1, 2, 3))[:, 0].detach().cpu().numpy()       # (C1, 7, 7)
    phi = unit(model.temporal_w @ model.tbasis.T, (1,)).detach().cpu().numpy()     # (C1, D), lag 0 first
    g   = model.gain.detach().cpu().numpy()
    K   = unit(model.layer2.weight, (1, 2, 3)).detach().cpu().numpy()              # (C2, C1, 5, 5)
    C2, C1 = K.shape[:2]
    sp = np.stack([[g[c] * convolve2d(K[m, c], S[c], mode="full") for c in range(C1)]
                   for m in range(C2)])                                            # (C2, C1, 11, 11)
    return np.einsum("mcyx,cd->mdyx", sp, phi)                                     # (C2, D, 11, 11)

rf = layer2_st_rf(model)
lag_ms = np.arange(rf.shape[1]) * 1000 / info.fps
half = (rf.shape[-1] - 1) / 2 * info.deg_per_tap
fig, axs = plt.subplots(2, rf.shape[0], figsize=(2.6 * rf.shape[0], 6), sharey=True)
for m in range(rf.shape[0]):
    for row, (proj, lab) in enumerate(((rf[m].sum(1), "x"), (rf[m].sum(2), "y"))):
        v = np.abs(proj).max()
        axs[row, m].imshow(proj, cmap="RdBu_r", vmin=-v, vmax=v, aspect="auto", origin="lower",
                           extent=[-half, half, lag_ms[0], lag_ms[-1]])
        axs[row, m].set_xlabel(f"{lab} [deg]")
    axs[0, m].set_title(f"2,{m+1}")
axs[0, 0].set_ylabel("lag [ms]"); axs[1, 0].set_ylabel("lag [ms]")
fig.tight_layout(); plt.show()

In [ ]:
from motionnet.train.loop import compute_v_std
valid = slice(info.gray_frames + model.max_delay - 1, None)
v_std = compute_v_std(data.train, valid, device)

model.eval()
preds, tgts, pools = [], [], []
with torch.no_grad():
    for vb in batches(data.val, 32):
        p = model(vb["movie"].to(device))[:, valid]
        preds.append(p); tgts.append(vb["vel_deg_s"].to(device)[:, valid] / v_std)
        B = vb["movie"].shape[0]
        pools.append(model._pool.reshape(B, -1, model._pool.shape[-1])[:, valid])
P_, T_, Y = torch.cat(preds).reshape(-1, 2), torch.cat(tgts).reshape(-1, 2), torch.cat(pools).reshape(-1, model.num_subunits_2)

# 1. bias hypothesis: each unit's constant (mean-activity) push on the output
R = unit(model.readout, (1,)).detach()
offset = model.out_gain * Y.mean(0)[:, None] * R.T              # (C2, 2)
print(pd.DataFrame(offset.cpu().numpy(), columns=["dx", "dy"],
                   index=[f"2,{m+1}" for m in range(len(offset))]).round(3))
print("net offset:", offset.sum(0).cpu().numpy().round(3))

# 2. asymmetry: is +v_x estimated worse than -v_x? (same check for y as control)
for k, name in enumerate(("v_x", "v_y")):
    for sgn, lab in ((1, "+"), (-1, "-")):
        m = (sgn * T_[:, k]) > 0.5                              # clearly moving that way (>0.5 s.d.)
        slope = (P_[m, k] * T_[m, k]).sum() / T_[m, k].pow(2).sum()
        print(f"{lab}{name}: gain {slope:.2f}   rmse {(P_[m, k] - T_[m, k]).pow(2).mean().sqrt():.2f}")

In [ ]:
def st_dsi(rf2d):
    """rf2d: (lag, space). +1/-1 = all energy for one direction of motion; 0 = separable/non-DS."""
    F = np.abs(np.fft.fftshift(np.fft.fft2(rf2d))) ** 2
    w = np.fft.fftshift(np.fft.fftfreq(rf2d.shape[0]))[:, None]   # temporal frequency
    k = np.fft.fftshift(np.fft.fftfreq(rf2d.shape[1]))[None, :]   # spatial frequency
    pos, neg = F[(w * k) > 0].sum(), F[(w * k) < 0].sum()
    return (pos - neg) / (pos + neg)

rows = []
for m in range(rf.shape[0]):
    rows.append(dict(unit=f"2,{m+1}", dsi_x=st_dsi(rf[m].sum(1)), dsi_y=st_dsi(rf[m].sum(2))))
print(pd.DataFrame(rows).set_index("unit").round(2))